# 02 - Player Analysis & Career Data

This notebook focuses on analyzing player careers, career statistics, and identifying players with Primavera experience.

**Prerequisites**: Complete `01_data_exploration.ipynb` first, or run `scripts/01_fetch_data.py` and `scripts/02_player_profiling.py`


## Setup & Data Loading

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import modules
from src.ingestion import DataFetcher
from src.features import CareerAnalyzer, DataProcessor
from src.utils import DataManager
from src.config import KNOWN_COMPETITIONS

import pandas as pd
import numpy as np
from tqdm import tqdm
from pprint import pprint

print("✓ Modules imported successfully")


In [ ]:
# Initialize utilities
manager = DataManager()
fetcher = DataFetcher()
analyzer = CareerAnalyzer()

# Load players data (from step 01 or from scripts)
try:
    df_players = manager.load_pickle('data/processed/players_profiles.pkl')
    print(f"✓ Loaded {len(df_players)} players from cache")
except FileNotFoundError:
    print("⚠ Players profiles not found. Run 01_data_exploration.ipynb first or run scripts/01_fetch_data.py and scripts/02_player_profiling.py")
    df_players = None

if df_players is not None:
    print(f"\nDataFrame shape: {df_players.shape}")
    print(f"Columns: {df_players.columns.tolist()}")
    print("\nFirst few rows:")
    print(df_players.head())


## Step 1: Explore Role Distribution

In [ ]:
# Analyze role distribution
if df_players is not None:
    role_distribution = df_players['role'].value_counts()
    print("Player Distribution by Role:")
    print(role_distribution)
    print(f"\nTotal roles: {len(role_distribution)}")
    
    # Players by team
    team_distribution = df_players['team_name'].value_counts()
    print(f"\nPlayers count by team (top 10):")
    print(team_distribution.head(10))


## Step 2: Analyze Single Player Career

In [ ]:
# Example: Analyze career for a single player
if df_players is not None and len(df_players) > 0:
    # Pick a player with interesting career
    sample_player = df_players.iloc[0]
    player_name = sample_player['name']
    player_wyid = sample_player['wyId']
    
    print(f"Analyzing career for: {player_name} (WyId: {player_wyid})")
    
    # Fetch raw career data
    career_data = fetcher.fetch_player_career(player_wyid)
    
    if career_data:
        # Enrich career data with readable names
        enriched_career = analyzer.enrich_career_data(career_data)
        
        # Calculate statistics
        career_stats = analyzer.calculate_career_statistics(career_data)
        
        print(f"\n✓ Career Statistics:")
        pprint(career_stats)
        
        # Show career as dataframe
        if enriched_career:
            career_df = pd.DataFrame(enriched_career)
            print(f"\nCareer Timeline:")
            print(career_df.head(10))
    else:
        print("No career data available for this player")


## Step 3: Identify Primavera Players

In [ ]:
# Function to check if player has Primavera experience
def check_primavera_experience(player_wy_id):
    """Check if player has played in Campionato Primavera 1 (competition ID: 516)"""
    career = fetcher.fetch_player_career(player_wy_id)
    
    if career:
        for entry in career:
            if entry.get('competitionId') == 516:  # Primavera 1
                return True
    return False

# Analyze a sample of players for Primavera experience
if df_players is not None and len(df_players) > 0:
    sample_size = min(50, len(df_players))  # Sample 50 players
    sample_wyids = df_players['wyId'].head(sample_size).tolist()
    
    print(f"Checking {sample_size} players for Primavera experience...")
    primavera_players = []
    
    for wyid in tqdm(sample_wyids):
        if check_primavera_experience(wyid):
            primavera_players.append(wyid)
    
    print(f"\n✓ Found {len(primavera_players)} players with Primavera experience out of {sample_size}")
    print(f"  Percentage: {len(primavera_players) / sample_size * 100:.1f}%")
    
    # Show Primavera players
    df_primavera = df_players[df_players['wyId'].isin(primavera_players)]
    print(f"\nPrimavera Players Sample:")
    print(df_primavera[['name', 'team_name', 'role']].head(10))


## Summary

This notebook demonstrates:
- ✓ Loading player profiles data
- ✓ Identifying players with Primavera experience
- ✓ Analyzing individual player careers
- ✓ Analyzing role/position distributions

**Next Step**: Run `03_advanced_stats_analysis.ipynb` to fetch and analyze advanced match statistics.
